In [1]:
from pathlib import Path

script = '''import pandas as pd

INPUT_FILE = "/home/sina/bigdata-lab/storage_lab/metadata/metadata_partition_date.csv"
OUTPUT_FILE = "/home/sina/bigdata-lab/storage_lab/metadata/metadata_partition_date.parquet"


def convert_to_parquet():
    df = pd.read_csv(INPUT_FILE)

    df.to_parquet(
        OUTPUT_FILE,
        engine="pyarrow",
        index=False
    )

    print(f"Parquet created: {OUTPUT_FILE}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")


if __name__ == "__main__":
    convert_to_parquet()
'''

path = Path("/home/sina/bigdata-lab/storage_lab/scripts/convert_metadata_to_parquet.py")
path.write_text(script)

print(f"Created: {path}")

Created: /home/sina/bigdata-lab/storage_lab/scripts/convert_metadata_to_parquet.py


## File Check

In [2]:
import pandas as pd

df = pd.read_csv(
    "/home/sina/bigdata-lab/storage_lab/metadata/metadata_partition_date.csv",
    low_memory=False,
    parse_dates=["ModificationTimestamp"]
)

FileNotFoundError: [Errno 2] No such file or directory: '/home/sina/bigdata-lab/storage_lab/metadata/metadata_partition_date.csv'

In [ ]:
import pandas as pd

file = "/home/sina/bigdata-lab/storage_lab/metadata/metadata_partition_date.parquet"

df = pd.read_parquet(file)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 2 rows:")
display(df.head(2))

# 1. Metadata Schema & Data Quality Validation

This section validates the analytical metadata dataset before performing storage analysis.

The validation process covers the following areas:

### 1.1 Basic Dataset Validation

### 1.2 Missing Value Validation

### 1.3 Duplicate Record Validation

### 1.4 Data Type Validation

### 1.5 File Metadata Validation

### 1.6 Partition Metadata Validation

### 1.7 Date & Time Validation

### 1.8 Path & Structure Validation

### 1.9 Logical Consistency Validation

### 1.10 Validation Summary

---

## Validation Result

The validated Parquet dataset will be used as the trusted input for the subsequent storage analysis.

### 1.1 Basic Dataset Validation
- Row count
- Column count
- Column names
- Dataset structure
- Dupplicate Check
- Sample records

In [ ]:
import pandas as pd

FILE = "/home/sina/bigdata-lab/storage_lab/metadata/metadata_partition_date.parquet"

df = pd.read_parquet(FILE)

print("=" * 70)
print("METADATA PARQUET VALIDATION")
print("=" * 70)

print(f"\nRows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

print("\n--- Schema ---")
print(df.dtypes)

print("\n--- Missing Values ---")
print(df.isna().sum())

print("\n--- Duplicate Rows ---")
print(f"Duplicates: {df.duplicated().sum():,}")

print("\n--- FileCount ---")
print(df["FileCount"].value_counts(dropna=False))

print("\n--- FileFormat ---")
print(df["FileFormat"].value_counts(dropna=False))

print("\n--- Zone ---")
print(df["Zone"].value_counts(dropna=False))

print("\n--- Partition Status ---")
print(df["IsPartitioned"].value_counts(dropna=False))

print("\n--- Sample ---")
display(df.head(5))

### 1.2 Missing Value Validation
- Missing values by column
- Missing-value percentage
- Identification of expected vs unexpected missing values

In [ ]:
# Expected vs Unexpected Missing Values

expected_missing = {
    "HourlyPartitionName": df["HasHourlyPartition"].eq(False),
    "PartitionColumn": ~df["IsPartitioned"],
    "PartitionName": ~df["IsPartitioned"],
    "PartitionPattern": ~df["IsPartitioned"]
}

print("=" * 70)
print("EXPECTED vs UNEXPECTED MISSING VALUES")
print("=" * 70)

for column, expected_condition in expected_missing.items():

    missing_mask = df[column].isna()

    expected = (missing_mask & expected_condition).sum()
    unexpected = (missing_mask & ~expected_condition).sum()

    print(f"\n{column}")
    print(f"  Expected Missing   : {expected:,}")
    print(f"  Unexpected Missing : {unexpected:,}")

### 1.3 Duplicate Record Validation
- Full-row duplicates
- Duplicate file paths
- Duplicate metadata records

In [ ]:
print("=" * 70)
print("DUPLICATE RECORD VALIDATION")
print("=" * 70)

full_duplicates = df.duplicated().sum()

print(f"\nFull-row duplicates: {full_duplicates:,}")

### 1.4 Data Type Validation
- Numeric columns
- Boolean columns
- Date and timestamp columns
- String columns
- Unexpected data types

In [ ]:
expected_types = {
    "TotalSizeBytes": "numeric",
    "FileCount": "numeric",
    "Year": "numeric",
    "Month": "numeric",
    "Day": "numeric",
    "Hour": "numeric",
    "IsPartitioned": "boolean",
    "HasHourlyPartition": "boolean",
    "ModificationTimestamp": "datetime"
}

print("DATA TYPE VALIDATION")
print("=" * 70)

for column, expected in expected_types.items():
    if expected == "numeric":
        valid = pd.api.types.is_numeric_dtype(df[column])
    elif expected == "boolean":
        valid = pd.api.types.is_bool_dtype(df[column])
    elif expected == "datetime":
        valid = pd.api.types.is_datetime64_any_dtype(df[column])
    else:
        valid = False

    print(f"{column:25} : {'PASS' if valid else 'FAIL'}")

### 1.5 File Metadata Validation
- File size validity
- File count validity
- File format consistency
- File name consistency
- Empty-directory records

In [ ]:
print("FILE METADATA VALIDATION")
print("=" * 70)

print(f"Negative FileSize      : {(df['TotalSizeBytes'] < 0).sum():,}")
print(f"Negative FileCount     : {(df['FileCount'] < 0).sum():,}")
print(f"Zero-size files        : {((df['TotalSizeBytes'] == 0) & (df['FileCount'] > 0)).sum():,}")
print(f"Invalid FileCount      : {(~df['FileCount'].isin([0, 1])).sum():,}")
print(f"Missing FileFormat     : {df['FileFormat'].isna().sum():,}")

file_rows = df["FullPath"].notna() & (df["FullPath"] != "")

print(f"File rows              : {file_rows.sum():,}")
print(f"FileName missing       : {df.loc[file_rows, 'FileName'].isna().sum():,}")

### 1.6 Partition Metadata Validation
- Partition flag consistency
- Partition column consistency
- Partition name consistency
- Partition pattern consistency
- Hourly partition consistency


In [ ]:
print("PARTITION METADATA VALIDATION")
print("=" * 70)

partitioned = df["IsPartitioned"] == True
non_partitioned = ~partitioned

print(f"Partitioned records                : {partitioned.sum():,}")
print(f"Non-partitioned records            : {non_partitioned.sum():,}")

print(f"PartitionColumn missing             : {df.loc[partitioned, 'PartitionColumn'].isna().sum():,}")
print(f"PartitionName missing               : {df.loc[partitioned, 'PartitionName'].isna().sum():,}")
print(f"PartitionPattern missing            : {df.loc[partitioned, 'PartitionPattern'].isna().sum():,}")

print(f"Unexpected PartitionColumn          : {df.loc[non_partitioned, 'PartitionColumn'].notna().sum():,}")
print(f"Unexpected PartitionName            : {df.loc[non_partitioned, 'PartitionName'].notna().sum():,}")
print(f"Unexpected PartitionPattern         : {df.loc[non_partitioned, 'PartitionPattern'].notna().sum():,}")

hourly = df["HasHourlyPartition"] == True

print(f"\nHourly-partitioned records          : {hourly.sum():,}")
print(f"HourlyPartitionName missing         : {df.loc[hourly, 'HourlyPartitionName'].isna().sum():,}")

### 1.7 Date & Time Validation
- Year / Month / Day consistency
- YearMonth consistency
- Hour consistency
- Modification date consistency
- Partition date consistency

In [ ]:
print("DATE & TIME VALIDATION")
print("=" * 70)

print(f"Invalid Month   : {(~df['Month'].between(1, 12)).sum():,}")
print(f"Invalid Day     : {(~df['Day'].between(1, 31)).sum():,}")
print(f"Invalid Hour    : {(~df['Hour'].between(0, 23)).sum():,}")

year_month_check = (
    df["Year"].astype("string") + "-" +
    df["Month"].astype(str).str.zfill(2)
)

print(f"Invalid YearMonth : {(df['YearMonth'] != year_month_check).sum():,}")

### 1.8 Path & Structure Validation
- Source consistency
- Zone consistency
- Table name consistency
- ParentPath consistency
- FullPath consistency
- FileName consistency

In [ ]:
print("PATH & STRUCTURE VALIDATION")
print("=" * 70)

source_check = df.apply(
    lambda r: r["Source"] in r["FullPath"] if pd.notna(r["FullPath"]) else True,
    axis=1
)

table_check = df.apply(
    lambda r: r["TableName"] in r["FullPath"] if pd.notna(r["FullPath"]) else True,
    axis=1
)

filename_check = df.apply(
    lambda r: (
        r["FileName"] in r["FullPath"]
        if pd.notna(r["FileName"]) and pd.notna(r["FullPath"])
        else True
    ),
    axis=1
)

print(f"Invalid Source in path     : {(~source_check).sum():,}")
print(f"Invalid TableName in path  : {(~table_check).sum():,}")
print(f"Invalid FileName in path   : {(~filename_check).sum():,}")

### 1.9 Logical Consistency Validation
- Cross-column consistency checks
- FileCount vs file-level records
- FileSize vs FileCount relationships
- Partition-related logical rules
- Identification of contradictory records

In [ ]:
print("LOGICAL CONSISTENCY VALIDATION")
print("=" * 70)

checks = {
    "FileCount=1 for files":
        ((df["Type"] != "EMPTY_DIRECTORY") & (df["FileCount"] != 1)).sum(),

    "FileCount=0 for empty directories":
        ((df["Type"] == "EMPTY_DIRECTORY") & (df["FileCount"] != 0)).sum(),

    "Empty directories have no FileName":
        ((df["Type"] == "EMPTY_DIRECTORY") & df["FileName"].notna()).sum(),

    "Partition flag consistency":
        ((df["IsPartitioned"] == False) &
         df["PartitionColumn"].notna()).sum(),

    "Hourly flag consistency":
        ((df["HasHourlyPartition"] == False) &
         df["HourlyPartitionName"].notna()).sum()
}

for check, failures in checks.items():
    print(f"{check:45} : {failures:,}")

### 1.10 Validation Summary
- Total validation checks
- Passed checks
- Failed checks
- Warnings
- Data quality issues requiring attention

In [ ]:
print("=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

summary = {
    "Full-row duplicates": full_duplicates,
    "Duplicate FullPath": duplicate_fullpath,
    "Negative file sizes": (df["TotalSizeBytes"] < 0).sum(),
    "Negative file counts": (df["FileCount"] < 0).sum(),
    "Invalid months": (~df["Month"].between(1, 12)).sum(),
    "Invalid days": (~df["Day"].between(1, 31)).sum(),
    "Invalid hours": (~df["Hour"].between(0, 23)).sum(),
    "Invalid YearMonth": (df["YearMonth"] != year_month_check).sum(),
}

display(
    pd.DataFrame.from_dict(
        summary,
        orient="index",
        columns=["IssueCount"]
    )
)

## Validation Result

The validated Parquet dataset will be used as the trusted input for the subsequent storage analysis.

In [ ]:
validation_issues = sum(summary.values())

print("=" * 70)
print("FINAL VALIDATION RESULT")
print("=" * 70)

if validation_issues == 0:
    print("PASS - No validation issues detected.")
else:
    print(f"REVIEW REQUIRED - {validation_issues:,} validation issues detected.")